In [1]:
!pip install wfdb torchmetrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 79.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you h

In [2]:
import pandas as pd
import numpy as np
import wfdb
import ast
import torch
from sklearn.preprocessing import MultiLabelBinarizer
import torchmetrics
import os

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
if device.type == 'cpu':
    from config import PARENT_PATH, CSV_PATH, BEST_MODEL_SAVE_DIR
else:
    CSV_PATH = "/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/"

In [6]:
if device.type == 'cuda':
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
if device.type == 'cuda':
    !time unzip -q "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip" -d "/content/"
    !ls -lah /content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3 | head


real	0m16.886s
user	0m7.760s
sys	0m3.080s
total 16M
drwxrwxrwx  3 root root 4.0K Mar 12 04:23 .
drwxr-xr-x  1 root root 4.0K Mar 17 20:48 ..
-rw-rw-rw-  1 root root 1.4K Mar  9 15:42 example_physionet.py
-rw-rw-rw-  1 root root  15K Mar  9 15:42 LICENSE.txt
-rw-rw-rw-  1 root root 6.3M Mar  9 15:42 ptbxl_database.csv
-rw-rw-rw-  1 root root 6.0K Mar  9 15:42 ptbxl_v102_changelog.txt
-rw-rw-rw-  1 root root 7.3K Mar  9 15:42 ptbxl_v103_changelog.txt
-rw-rw-rw-  1 root root 1.1M Mar  9 15:42 RECORDS
drwxrwxrwx 24 root root 4.0K Mar  9 15:48 records100


In [8]:
if device.type == 'cuda':
    best_model_save_dir = '/content/drive/MyDrive/'
else:
    best_model_save_dir = BEST_MODEL_SAVE_DIR

In [9]:
def load_raw_data(df, sampling_rate):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(os.path.join(parent_path, f)) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(os.path.join(parent_path, f)) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

if device.type == 'cuda':
    parent_path = '/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
    # parent_path = PARENT_PATH
else:
    #parent_path = os.curdir
    parent_path = PARENT_PATH
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(os.path.join(CSV_PATH, 'ptbxl_database.csv'), index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(os.path.join(CSV_PATH, 'scp_statements.csv'), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

In [10]:
# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y.strat_fold != test_fold)]
y_train = Y[(Y.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y.strat_fold == test_fold)]
y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

In [11]:
mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(y_train)
y_test = mlb.transform(y_test)

In [12]:
X_train.shape, X_train.dtype, y_train.shape, y_train.dtype

((19601, 1000, 12), dtype('float64'), (19601, 5), dtype('int64'))

In [13]:
# just examining the data some
summed = np.sum(y_train, axis=1)
record_ids_with_no_labels_true = []
for id, class_sum in enumerate(summed):
    if class_sum == 0:
        record_ids_with_no_labels_true.append(id)
print(f'{len(record_ids_with_no_labels_true)} train examples have all labels as "0"')

371 train examples have all labels as "0"


In [14]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32).transpose(1,2)  # the 12 channels dimension is initially loaded in as last
y_train_tensor = torch.tensor(y_train,dtype=torch.float32)

X_test_tensor = torch.tensor(X_test,dtype=torch.float32).transpose(1,2)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32)

In [15]:
from torch.utils.data import TensorDataset,DataLoader
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [16]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.conv1 = nn.Conv1d(12, 32, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(2)

        # average the whole time series into 1 length to account for any time series length
        # while this discards temporal detail, the abnormalities are more concerned over the presence/strength of extracted features (representing the presence of abnormalities)
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):

        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.squeeze(-1)

        x = self.fc(x)

        return x

In [17]:
# count class frequencies in training set
pos_counts = y_train_tensor.sum(dim=0)
neg_counts = len(y_train_tensor) - pos_counts
pos_weight = (neg_counts / pos_counts).to(device)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# loss_fn = nn.BCEWithLogitsLoss()  

In [96]:
def train_model(model, optimizer, loss_fn, train_loader, test_loader, best_model_save_path, num_epochs=10, scheduler=None):
    best_loss = float('inf')  # start with infinity
    for epoch in range(num_epochs):
        running_train_loss = 0.0
        model.train()

        for x, y in train_loader:

            x = x.to(device)
            y = y.to(device)

            preds = model(x)

            loss = loss_fn(preds, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            running_train_loss += loss.item() * x.size(0)
        epoch_train_loss = running_train_loss / len(train_loader.dataset)

        running_test_loss = 0.0
        model.eval()
        with torch.no_grad():
            for x, y in test_loader:
                x = x.to(device)
                y = y.to(device)
                preds = model(x)
                loss = loss_fn(preds, y)
                running_test_loss += loss.item() * x.size(0)
            epoch_test_loss = running_test_loss / len(test_loader.dataset)

        print(f"Epoch {epoch+1}, Training Loss: {epoch_train_loss:.4f}, Testing Loss: {epoch_test_loss:.4f}")

        if epoch_test_loss < best_loss:
            best_loss = epoch_test_loss
            torch.save(model.state_dict(), best_model_save_path)

In [ ]:
def run_inference(model, dataloader):
    model.eval()
    all_test_preds, all_test_labels = [], []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_test_preds.append(preds.cpu())
            all_test_labels.append(y.cpu())

    # Concatenate all batches
    all_test_preds = torch.cat(all_test_preds, dim=0)
    all_test_labels = torch.cat(all_test_labels, dim=0)

    return all_test_preds, all_test_labels

In [20]:
model = CNN(num_classes=y_train_tensor.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_model(model, optimizer, loss_fn, train_loader, test_loader, os.path.join(best_model_save_dir, 'cnn_best_model.pt'), num_epochs=30)

Epoch 1, Training Loss: 0.6878, Testing Loss: 0.6331
Epoch 2, Training Loss: 0.6029, Testing Loss: 0.6798
Epoch 3, Training Loss: 0.5672, Testing Loss: 0.5849
Epoch 4, Training Loss: 0.5469, Testing Loss: 0.5816
Epoch 5, Training Loss: 0.5319, Testing Loss: 0.5967
Epoch 6, Training Loss: 0.5240, Testing Loss: 0.5999
Epoch 7, Training Loss: 0.5147, Testing Loss: 0.5694
Epoch 8, Training Loss: 0.5069, Testing Loss: 0.5630
Epoch 9, Training Loss: 0.4988, Testing Loss: 0.5780
Epoch 10, Training Loss: 0.4925, Testing Loss: 0.5496
Epoch 11, Training Loss: 0.4844, Testing Loss: 0.5591
Epoch 12, Training Loss: 0.4809, Testing Loss: 0.5918
Epoch 13, Training Loss: 0.4783, Testing Loss: 0.5445
Epoch 14, Training Loss: 0.4699, Testing Loss: 0.5524
Epoch 15, Training Loss: 0.4669, Testing Loss: 0.5514
Epoch 16, Training Loss: 0.4614, Testing Loss: 0.5568
Epoch 17, Training Loss: 0.4576, Testing Loss: 0.5464
Epoch 18, Training Loss: 0.4539, Testing Loss: 0.5736
Epoch 19, Training Loss: 0.4492, Test

In [21]:
all_test_preds, all_test_labels = run_inference(model, test_loader)

In [22]:
# Per-class accuracy
def calc_per_class_accs(all_test_preds, all_test_labels):
    per_class_acc = (all_test_preds == all_test_labels).float().mean(dim=0)  # [num_classes]
    # Print results
    for idx, acc in enumerate(per_class_acc):
        print(f"Class {mlb.classes_[idx]} accuracy: {acc.item():.3f}")
    print('\n')

In [23]:
from sklearn.metrics import f1_score
def calc_f1_scores(all_test_labels, all_test_preds, average=None):
    f1_scores = f1_score(all_test_labels, all_test_preds, average=None)
    for idx, score in enumerate(f1_scores):
        print(f"Class {mlb.classes_[idx]} F1-score: {score:.3f}")
    print('\n')

In [24]:
from sklearn.metrics import roc_auc_score

def calc_roc_auc_scores(all_test_labels, all_test_preds):
    num_classes = all_test_labels.shape[1]
    aucs = []

    for i in range(num_classes):
        try:
            auc = roc_auc_score(all_test_labels[:, i], all_test_preds[:, i])
        except ValueError:
            # happens if class i has no positive samples in test set
            auc = float('nan')
        aucs.append(auc)

    for idx, auc in enumerate(aucs):
        print(f"Class {mlb.classes_[idx]} AUROC: {auc:.3f}")
    print('\n')

In [25]:
print('-----CNN Test Evals-----')
calc_per_class_accs(all_test_labels, all_test_preds)
calc_f1_scores(all_test_labels, all_test_preds)
calc_roc_auc_scores(all_test_labels, all_test_preds)

-----CNN Test Evals-----
Class CD accuracy: 0.858
Class HYP accuracy: 0.882
Class MI accuracy: 0.856
Class NORM accuracy: 0.851
Class STTC accuracy: 0.852


Class CD F1-score: 0.721
Class HYP F1-score: 0.582
Class MI F1-score: 0.733
Class NORM F1-score: 0.849
Class STTC F1-score: 0.727


Class CD AUROC: 0.843
Class HYP AUROC: 0.799
Class MI AUROC: 0.834
Class NORM AUROC: 0.862
Class STTC AUROC: 0.845




### --- GRU Model ---

In [84]:
# GRU wants batched input as (B, L, H_in), unlike for our CNN model
X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.float32)

X_test_tensor = torch.tensor(X_test,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32)

In [85]:
from torch.utils.data import TensorDataset,DataLoader
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [87]:
import torch.nn as nn
import torch.nn.functional as F

# following kaggle tutorial here https://www.kaggle.com/code/fanbyprinciple/learning-pytorch-3-coding-an-rnn-gru-lstm
class ECGGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, sequence_length, bidirectional, dropout_pct=0.2):
        super(ECGGRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        self.dropout_pct = dropout_pct

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bias=True,
            batch_first=True,
            bidirectional=bidirectional,
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(hidden_size * self.num_directions, num_classes)
        self.dropout = nn.Dropout(dropout_pct)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers * self.num_directions, x.size(0), self.hidden_size).to(device)
        out, _ = self.gru(x, h0)
        out = out.permute(0, 2, 1)  # reshape for pooling [Batch, 128, 1000]; hidden features treated as "channels"
        out = self.global_pool(out)  # Global Average Pooling step - looks for the "average presence of features across the sequence"
        out = out.squeeze(-1)  # need to flatten the last dim
        if self.dropout_pct > 0:
            out = self.dropout(out)  # for extra regularization
        out = self.fc1(out)
        return out

In [88]:
hidden_size = 64
num_layers = 4
bidirectional = True
dropout_pct = 0.2

model = ECGGRU(
    input_size=12,
    hidden_size=hidden_size,
    num_layers=num_layers,
    num_classes=5,
    sequence_length=1000,
    bidirectional=bidirectional,  # keep False on CPU, True only on GPU
    dropout_pct=dropout_pct
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model_pt_name = f'gru_bi{bidirectional}_hidden{hidden_size}_layers{num_layers}_dropout{dropout_pct}.pt'

In [89]:
train_model(model, optimizer, loss_fn, train_loader, test_loader, os.path.join(best_model_save_dir, model_pt_name), num_epochs=20)

Epoch 1, Training Loss: 0.7884, Testing Loss: 0.6950
Epoch 2, Training Loss: 0.6186, Testing Loss: 0.6409
Epoch 3, Training Loss: 0.5703, Testing Loss: 0.6361
Epoch 4, Training Loss: 0.5417, Testing Loss: 0.6000
Epoch 5, Training Loss: 0.5179, Testing Loss: 0.5685
Epoch 6, Training Loss: 0.4997, Testing Loss: 0.5582
Epoch 7, Training Loss: 0.4855, Testing Loss: 0.5517
Epoch 8, Training Loss: 0.4692, Testing Loss: 0.5513
Epoch 9, Training Loss: 0.4565, Testing Loss: 0.5744
Epoch 10, Training Loss: 0.4439, Testing Loss: 0.5353
Epoch 11, Training Loss: 0.4328, Testing Loss: 0.5405
Epoch 12, Training Loss: 0.4194, Testing Loss: 0.5332
Epoch 13, Training Loss: 0.4084, Testing Loss: 0.5658
Epoch 14, Training Loss: 0.3948, Testing Loss: 0.5588
Epoch 15, Training Loss: 0.3812, Testing Loss: 0.5485
Epoch 16, Training Loss: 0.3711, Testing Loss: 0.5851
Epoch 17, Training Loss: 0.3558, Testing Loss: 0.5862
Epoch 18, Training Loss: 0.3412, Testing Loss: 0.5768
Epoch 19, Training Loss: 0.3308, Test

In [92]:
# load the best model from GRU training
best_gru_model = ECGGRU(
    input_size=12,
    hidden_size=hidden_size,      # match what was trained
    num_layers=num_layers,        # match what was trained
    num_classes=5,
    sequence_length=1000,
    bidirectional=bidirectional,
    dropout_pct=dropout_pct
).to(device)

checkpoint_path = os.path.join(best_model_save_dir, model_pt_name)
best_gru_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))

<All keys matched successfully>

In [93]:
all_test_preds, all_test_labels = run_inference(best_gru_model, test_loader)

In [94]:
def calc_f1_scores(all_test_labels, all_test_preds, average=None):
    # Handle torch tensors
    if hasattr(all_test_labels, 'detach'):
        all_test_labels = all_test_labels.detach().cpu().numpy()
    if hasattr(all_test_preds, 'detach'):
        all_test_preds = all_test_preds.detach().cpu().numpy()
    
    f1_scores = f1_score(all_test_labels, all_test_preds, average=None)
    for idx, score in enumerate(f1_scores):
        print(f"Class {mlb.classes_[idx]} F1-score: {score:.3f}")
    print('\n')

In [95]:
print('-----GRU Test Evals-----')
calc_per_class_accs(all_test_labels, all_test_preds)
calc_f1_scores(all_test_labels, all_test_preds)
calc_roc_auc_scores(all_test_labels, all_test_preds)

-----GRU Test Evals-----
Class CD accuracy: 0.845
Class HYP accuracy: 0.815
Class MI accuracy: 0.856
Class NORM accuracy: 0.868
Class STTC accuracy: 0.850


Class CD F1-score: 0.712
Class HYP F1-score: 0.514
Class MI F1-score: 0.743
Class NORM F1-score: 0.857
Class STTC F1-score: 0.731


Class CD AUROC: 0.847
Class HYP AUROC: 0.817
Class MI AUROC: 0.848
Class NORM AUROC: 0.872
Class STTC AUROC: 0.854




### CNN + GRU
1D-CNN: Filters noise and finds the "shape" of the heartbeat.

GRU: Analyzes the sequence of data

Global Average Pooling: Summarizes the 250 steps into one vector.

Linear Layer: Outputs probabilities for the 5 superclasses.

In [100]:
class ECGConvGRU(nn.Module):
    def __init__(self, input_size=12, hidden_size=64, num_layers=2, num_classes=5):
        super(ECGConvGRU, self).__init__()
        
        # 1. 1D-CNN Front-end
        self.cnn = nn.Sequential(
            nn.Conv1d(input_size, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        
        # 2. GRU Backbone
        self.gru = nn.GRU(
            input_size=128, 
            hidden_size=hidden_size,
            num_layers=num_layers,
            bias=True,
            batch_first=True,
            bidirectional=True
        )
        
        # 3. Global Pooling & Final Classifier
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        if x.shape[1] > x.shape[2]:  # if x is [Batch, 1000, 12], need to switch to [Batch, 12, 1000]
            x = x.permute(0, 2, 1)
        features = self.cnn(x)
        
        # GRU expects: [Batch, Seq_len, Features]
        features = features.permute(0, 2, 1)
        gru_out, _ = self.gru(features)
        
        # Pool across sequence dimension: [Batch, hidden*2, 1]
        pooled = self.global_pool(gru_out.permute(0, 2, 1)).squeeze(-1)
        
        return self.fc(pooled)

In [106]:
import torch.optim as optim

hidden_size = 64
num_layers = 2

learning_rate = 1e-3
epochs = 50

model = ECGConvGRU(
    hidden_size=hidden_size,
    num_layers=num_layers,
).to(device)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
# "OneCycleLR starts low, peaks in the middle, and goes very low at the end
# It helps the model "settle" into a local minimum for better generalization"
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=learning_rate, 
    steps_per_epoch=len(train_loader), 
    epochs=epochs
)
model_pt_name = f'cnn_gru_bi1_hidden{hidden_size}_layers{num_layers}.pt'

In [107]:
train_model(model, optimizer, loss_fn, train_loader, test_loader, os.path.join(best_model_save_dir, model_pt_name), num_epochs=epochs)

Epoch 1, Training Loss: 0.9212, Testing Loss: 0.8107
Epoch 2, Training Loss: 0.7529, Testing Loss: 0.7413
Epoch 3, Training Loss: 0.6857, Testing Loss: 0.6989
Epoch 4, Training Loss: 0.6513, Testing Loss: 0.6826
Epoch 5, Training Loss: 0.6327, Testing Loss: 0.6515
Epoch 6, Training Loss: 0.6178, Testing Loss: 0.6439
Epoch 7, Training Loss: 0.6022, Testing Loss: 0.6341
Epoch 8, Training Loss: 0.5906, Testing Loss: 0.6347
Epoch 9, Training Loss: 0.5818, Testing Loss: 0.6259
Epoch 10, Training Loss: 0.5720, Testing Loss: 0.6200
Epoch 11, Training Loss: 0.5619, Testing Loss: 0.6136
Epoch 12, Training Loss: 0.5564, Testing Loss: 0.6000
Epoch 13, Training Loss: 0.5474, Testing Loss: 0.6000
Epoch 14, Training Loss: 0.5437, Testing Loss: 0.5914
Epoch 15, Training Loss: 0.5357, Testing Loss: 0.5890
Epoch 16, Training Loss: 0.5306, Testing Loss: 0.5974
Epoch 17, Training Loss: 0.5254, Testing Loss: 0.5818
Epoch 18, Training Loss: 0.5202, Testing Loss: 0.5868
Epoch 19, Training Loss: 0.5156, Test

In [108]:
# load the best model from CNN-GRU training
best_cnn_gru_model = ECGConvGRU(
    hidden_size=hidden_size,
    num_layers=num_layers,
).to(device)

checkpoint_path = os.path.join(best_model_save_dir, model_pt_name)
best_cnn_gru_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))

<All keys matched successfully>

In [109]:
all_test_preds, all_test_labels = run_inference(best_cnn_gru_model, test_loader)

In [110]:
print('-----CNN-GRU Test Evals-----')
calc_per_class_accs(all_test_labels, all_test_preds)
calc_f1_scores(all_test_labels, all_test_preds)
calc_roc_auc_scores(all_test_labels, all_test_preds)

-----CNN-GRU Test Evals-----
Class CD accuracy: 0.864
Class HYP accuracy: 0.839
Class MI accuracy: 0.824
Class NORM accuracy: 0.854
Class STTC accuracy: 0.838


Class CD F1-score: 0.725
Class HYP F1-score: 0.532
Class MI F1-score: 0.697
Class NORM F1-score: 0.836
Class STTC F1-score: 0.719


Class CD AUROC: 0.840
Class HYP AUROC: 0.808
Class MI AUROC: 0.819
Class NORM AUROC: 0.854
Class STTC AUROC: 0.850




### CNN + Attention
1D-CNN: Filters noise and finds the "shape" of the heartbeat.

Sinusoidal Positional Encoding: Tells the model "when" things are happening.

Transformer Encoder (Self-Attention): Compares every heartbeat to every other heartbeat to find rhythm abnormalities.

Global Average Pooling: Summarizes the 250 steps into one vector.

Linear Layer: Outputs probabilities for the 5 superclasses.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class ECGTransformer(nn.Module):
    def __init__(self, input_size=12, num_classes=5, embed_dim=128, num_heads=8, num_layers=2):
        super(ECGTransformer, self).__init__()
        
        # 1. 1D-CNN Front-end
        self.cnn = nn.Sequential(
            nn.Conv1d(input_size, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2), # Seq length: 1000 -> 500
            nn.Conv1d(64, embed_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(),
            nn.MaxPool1d(2)  # Seq length: 500 -> 250
        )
        
        # 2. Sinusoidal Positional Encoding
        self.pos_encoder = PositionalEncoding(embed_dim, max_len=250)
        
        # 3. Transformer Encoder (Self-Attention)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, batch_first=True, dropout=0.2
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 4. Global Average Pooling & 5. Final Linear Layer
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        if x.shape[1] > x.shape[2]:  # if x is [Batch, 1000, 12], need to switch to [Batch, 12, 1000] for CNN
            x = x.permute(0, 2, 1)
        features = self.cnn(x)
        
        # Prep for Transformer: [Batch, 250, 128]
        features = features.permute(0, 2, 1)
        features = self.pos_encoder(features)
        
        # Self-Attention
        att_out = self.transformer(features)
        
        # Pool across sequence dimension: [Batch, 128, 1]
        pooled = self.global_pool(att_out.permute(0, 2, 1)).squeeze(-1)
        
        return self.fc(pooled)

In [ ]:
import torch.optim as optim

embed_dim = 128
num_heads = 8
num_layers = 2

learning_rate = 1e-3
epochs = 20

model = ECGTransformer(
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_layers=num_layers
).to(device)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
# "OneCycleLR starts low, peaks in the middle, and goes very low at the end
# It helps the model "settle" into a local minimum for better generalization"
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=learning_rate, 
    steps_per_epoch=len(train_loader), 
    epochs=epochs
)
model_pt_name = f'cnn_transformer_embed{embed_dim}_heads{num_heads}_layers{num_layers}.pt'

In [ ]:
train_model(model, optimizer, loss_fn, train_loader, test_loader, os.path.join(best_model_save_dir, model_pt_name), num_epochs=epochs)

In [ ]:
# load the best model from CNN-GRU training
best_cnn_transformer_model = ECGTransformer(
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_layers=num_layers
).to(device)

checkpoint_path = os.path.join(best_model_save_dir, model_pt_name)
best_cnn_transformer_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))

In [ ]:
all_test_preds, all_test_labels = run_inference(best_cnn_transformer_model, test_loader)

In [ ]:
print('-----CNN-Transformer Test Evals-----')
calc_per_class_accs(all_test_labels, all_test_preds)
calc_f1_scores(all_test_labels, all_test_preds)
calc_roc_auc_scores(all_test_labels, all_test_preds)